# Model Evaluation and Diagnostics

**Chapter 6: Regression Techniques for Soccer Analytics**

## Learning Objectives

- Master key regression performance metrics (R², RMSE, MAE)
- Understand the importance of train/test splits and validation sets
- Perform residual analysis to diagnose model problems
- Identify and handle influential observations
- Compare multiple models systematically
- Interpret model coefficients in soccer context


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


## Key Performance Metrics

For regression models, we have several metrics to evaluate performance:

| Metric | Formula | Interpretation | Range |
|--------|---------|----------------|-------|
| **R²** | 1 - (SS_res / SS_tot) | % of variance explained | 0 to 1 (higher better) |
| **RMSE** | √(mean(errors²)) | Average prediction error (same units) | 0 to ∞ (lower better) |
| **MAE** | mean(\|errors\|) | Average absolute error | 0 to ∞ (lower better) |
| **MAPE** | mean(\|errors/actual\|) × 100 | Average % error | 0 to ∞ (lower better) |


## Load Data: Match Outcomes

We'll use match data to predict goals scored.


In [ ]:
# Simulated match data
np.random.seed(42)
n_matches = 100

data = {
    'ShotsOnTarget': np.random.randint(3, 15, n_matches),
    'Possession': np.random.uniform(35, 65, n_matches),
    'xG': np.random.uniform(0.5, 3.5, n_matches),
    'Goals': np.random.randint(0, 5, n_matches)
}

df = pd.DataFrame(data)
# Add realistic correlation
df['Goals'] = (df['xG'] * 0.8 + df['ShotsOnTarget'] * 0.1 + 
               np.random.normal(0, 0.5, n_matches)).clip(0, 5).round().astype(int)

print(\"Match Dataset:\")
print(df.head(10))
print(f\"\
Dataset shape: {df.shape}\")
print(f\"\
Basic statistics:\")
print(df.describe())


## Train/Test Split: The Foundation of Evaluation

**Critical concept:** Never evaluate on training data!

- **Training set:** Used to fit the model
- **Test set:** Used to evaluate generalization
- **Validation set:** Used for hyperparameter tuning (optional)

**Why?** Training performance is optimistic. Test performance shows real-world capability.


In [ ]:
# Prepare features and target
features = ['ShotsOnTarget', 'Possession', 'xG']
target = 'Goals'

X = df[features]
y = df[target]

# Split: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f\"Training set: {len(X_train)} matches ({len(X_train)/len(df)*100:.0f}%)\")
print(f\"Test set: {len(X_test)} matches ({len(X_test)/len(df)*100:.0f}%)\")
print(f\"\
This ensures unbiased evaluation on unseen data.\")


## Build and Evaluate a Model


In [ ]:
# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(\"Model Performance:\")
print(f\"\
{'Metric':<15} {'Training':<15} {'Test':<15}\")
print(\"-\" * 45)
print(f\"{'R²':<15} {train_r2:<15.3f} {test_r2:<15.3f}\")
print(f\"{'RMSE':<15} {train_rmse:<15.3f} {test_rmse:<15.3f}\")
print(f\"{'MAE':<15} {train_mae:<15.3f} {test_mae:<15.3f}\")

if abs(train_r2 - test_r2) > 0.1:
    print(\"\
⚠️ Warning: Large gap between train and test performance suggests overfitting!\")
else:
    print(\"\
✅ Train and test performance are similar - good generalization!\")


## Interpreting Metrics

**R² = 0.85** means:
- 85% of variance in goals is explained by our features
- 15% is unexplained (random variation, missing features)

**RMSE = 0.6 goals** means:
- On average, predictions are off by 0.6 goals
- In soccer context: predicting 2.3 goals when actual is 2 or 3

**MAE = 0.5 goals** means:
- Average absolute error is half a goal
- More interpretable than RMSE for stakeholders


## Residual Analysis: Diagnosing Problems

**Residuals** = Actual - Predicted

A good model should have:
1. **Random scatter** around zero (no patterns)
2. **Constant variance** (homoscedasticity)
3. **Normal distribution** (for statistical inference)


In [ ]:
# Calculate residuals
residuals = y_test - y_test_pred

# Create residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Predicted
axes[0, 0].scatter(y_test_pred, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Goals')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs. Predicted Values')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals vs Actual
axes[0, 1].scatter(y_test, residuals, alpha=0.6)
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Actual Goals')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals vs. Actual Values')
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of residuals
axes[1, 0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')

# 4. Q-Q plot
from scipy import stats
stats.probplot(residuals, dist=\"norm\", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

print(\"Residual Analysis Checklist:\")
print(\"✓ Random scatter around zero? (No U-shape or pattern)\")
print(\"✓ Constant variance? (No funnel shape)\")
print(\"✓ Approximately normal? (Histogram bell-shaped, Q-Q plot linear)\")


## Identifying Influential Observations

**Influential points** can disproportionately affect the model. Let's find them!


In [ ]:
# Use statsmodels for influence diagnostics
X_train_sm = sm.add_constant(X_train)
model_sm = sm.OLS(y_train, X_train_sm).fit()

# Get influence measures
influence = model_sm.get_influence()
leverage = influence.hat_matrix_diag
cooks_d = influence.cooks_distance[0]

# Plot Cook's distance
plt.figure(figsize=(12, 6))
plt.stem(range(len(cooks_d)), cooks_d, markerfmt=',')
plt.axhline(y=4/len(X_train), color='r', linestyle='--', label='Threshold (4/n)')
plt.xlabel('Observation Index')
plt.ylabel(\"Cook's Distance\")
plt.title(\"Influential Observations (Cook's Distance)\")
plt.legend()
plt.tight_layout()
plt.show()

# Identify influential points
threshold = 4 / len(X_train)
influential_idx = np.where(cooks_d > threshold)[0]

if len(influential_idx) > 0:
    print(f\"Found {len(influential_idx)} influential observations:\")
    print(X_train.iloc[influential_idx])
    print(\"\
These points have high leverage on the model. Consider investigating them!\")
else:
    print(\"No highly influential observations detected.\")


## Cross-Validation: Robust Evaluation

**Problem:** Single train/test split can be lucky or unlucky.

**Solution:** K-Fold Cross-Validation
- Split data into K folds
- Train on K-1 folds, test on 1
- Repeat K times
- Average the results


In [ ]:
# Perform 5-fold cross-validation
from sklearn.model_selection import cross_validate

model_cv = LinearRegression()

# Get multiple metrics
scoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
cv_results = cross_validate(model_cv, X, y, cv=5, scoring=scoring)

# Extract and display results
r2_scores = cv_results['test_r2']
rmse_scores = np.sqrt(-cv_results['test_neg_mean_squared_error'])
mae_scores = -cv_results['test_neg_mean_absolute_error']

print(\"5-Fold Cross-Validation Results:\")
print(f\"\
R² scores: {r2_scores}\")
print(f\"  Mean: {r2_scores.mean():.3f} (±{r2_scores.std():.3f})\")
print(f\"\
RMSE scores: {rmse_scores}\")
print(f\"  Mean: {rmse_scores.mean():.3f} (±{rmse_scores.std():.3f})\")
print(f\"\
MAE scores: {mae_scores}\")
print(f\"  Mean: {mae_scores.mean():.3f} (±{mae_scores.std():.3f})\")
print(f\"\
Cross-validation provides more reliable performance estimates!\")


## Model Comparison Framework

Let's compare multiple models systematically.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'KNN (K=5)': KNeighborsRegressor(n_neighbors=5),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42)
}

# Scale features for KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Compare models
results = []

for name, model in models.items():
    # Use scaled data for KNN, original for others
    X_use = X_scaled if 'KNN' in name else X
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_use, y, cv=5, scoring='r2')
    
    # Train on full training set and evaluate on test
    X_train_use = scaler.transform(X_train) if 'KNN' in name else X_train
    X_test_use = scaler.transform(X_test) if 'KNN' in name else X_test
    
    model.fit(X_train_use, y_train)
    test_r2 = model.score(X_test_use, y_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test_use)))
    
    results.append({
        'Model': name,
        'CV R² (mean)': cv_scores.mean(),
        'CV R² (std)': cv_scores.std(),
        'Test R²': test_r2,
        'Test RMSE': test_rmse
    })

# Display comparison
results_df = pd.DataFrame(results)
print(\"Model Comparison:\")
print(results_df.to_string(index=False))
print(f\"\
Best model by CV R²: {results_df.loc[results_df['CV R² (mean)'].idxmax(), 'Model']}\")
print(f\"Best model by Test R²: {results_df.loc[results_df['Test R²'].idxmax(), 'Model']}\")


## Visualize Model Comparison


In [ ]:
# Bar plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# R² comparison
axes[0].bar(results_df['Model'], results_df['Test R²'], alpha=0.7)
axes[0].set_ylabel('R² Score')
axes[0].set_title('Model Comparison: R² Score')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# RMSE comparison
axes[1].bar(results_df['Model'], results_df['Test RMSE'], alpha=0.7, color='coral')
axes[1].set_ylabel('RMSE')
axes[1].set_title('Model Comparison: RMSE')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Interpreting Coefficients

For linear models, coefficients tell us feature importance.


In [ ]:
# Get coefficients from linear model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': linear_model.coef_
}).sort_values('Coefficient', ascending=False)

print(\"Feature Coefficients:\")
print(coef_df.to_string(index=False))
print(f\"\
Intercept: {linear_model.intercept_:.3f}\")
print(f\"\
Interpretation:\")
for _, row in coef_df.iterrows():
    print(f\"  {row['Feature']}: +1 unit → {row['Coefficient']:+.3f} goals\")

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel('Coefficient Value')
plt.title('Feature Importance (Linear Regression Coefficients)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()


## Summary

In this notebook, we:

1. ✅ Mastered key regression metrics (R², RMSE, MAE)
2. ✅ Implemented proper train/test splits
3. ✅ Performed comprehensive residual analysis
4. ✅ Identified influential observations
5. ✅ Used cross-validation for robust evaluation
6. ✅ Compared multiple models systematically
7. ✅ Interpreted model coefficients

## Key Takeaways

- **Never evaluate on training data** - always use test set or cross-validation
- **Multiple metrics** give a complete picture (R², RMSE, MAE)
- **Residual analysis** reveals model problems
- **Cross-validation** provides more reliable estimates
- **Model comparison** should be systematic and data-driven
- **Interpretation matters** - coefficients tell the story

## Next Steps

In the next notebook, we'll apply everything to a **Practical Case Study** predicting match outcomes!


## Practice Exercises

1. **Validation Set**: Implement a train/validation/test split (60/20/20)
2. **Adjusted R²**: Calculate and compare with regular R²
3. **Prediction Intervals**: Add confidence intervals to predictions
4. **Feature Selection**: Use p-values to select significant features
5. **Learning Curves**: Plot training/test scores vs. dataset size
